# Task 1: News Topic Classifier Using BERT

### Problem Statement & Objective
The objective of this task is to fine-tune a pre-trained transformer model (`bert-base-uncased`) to automatically classify news headlines into their respective topic categories. This notebook covers text preprocessing, transfer learning/fine-tuning, model evaluation using accuracy and F1-score, and preparation for deployment via Gradio.

### Dataset
We utilize the **AG News Dataset** loaded from the Hugging Face Hub, which contains news articles categorized into four distinct classes: World, Sports, Business, and Sci/Tech.

In [3]:
# Install required libraries
!pip install -q transformers[torch] datasets evaluate scikit-learn accelerate gradio

import os
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report

## 1. Dataset Loading & Preprocessing

Here, we download the dataset using its namespaced repository ID to comply with Hugging Face conventions and load the pre-trained `bert-base-uncased` tokenizer. The input headlines are tokenized, padded, and truncated to ensure uniform sizing before being passed to the transformer backend.

In [6]:
# Load the dataset using the official namespace repository path
dataset = load_dataset("SetFit/ag_news")

# Load the tokenizer
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Define the tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Tokenize datasets and prepare train/validation subsets for quick demo training
tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1500))

## 2. Model Development & Training

We now initialize the `bert-base-uncased` sequence classification architecture configured with 4 target class outputs. We define custom metric evaluations (Accuracy and F1-score) and pass them along with our hyperparameters to the Hugging Face `Trainer` engine.

In [8]:
# Load the model with 4 distinct class labels
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)

# Set up evaluation metri cs
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

# Define training hyperparameters
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

# Initialize Trainer using the updated processing_class parameter
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Execute Fine-tuning
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.282943,0.251417,0.911333,0.911205
2,0.175931,0.246782,0.922000,0.922195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1250, training_loss=0.28472751083374026, metrics={'train_runtime': 584.6381, 'train_samples_per_second': 34.209, 'train_steps_per_second': 2.138, 'total_flos': 1315578900480000.0, 'train_loss': 0.28472751083374026, 'epoch': 2.0})

## 3. Model Evaluation

With training complete, we make inference predictions on our unseen test subset to calculate final performance metrics. A detailed classification report breaks down precision, recall, and our core target metrics: **Accuracy** and **F1-score**.

In [1]:
# Evaluate model predictions
predictions = trainer.predict(eval_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Print out evaluation matrix
print("Evaluation Summary:")
print(classification_report(labels, preds, target_names=["World", "Sports", "Business", "Sci/Tech"]))

# Save local copy of fine-tuned artifacts for immediate application hosting
model.save_pretrained("./fine-tuned-bert-news")
tokenizer.save_pretrained("./fine-tuned-bert-news")

## 4. Deployment Prep (Gradio Interactive Script)

To enable live interactive testing, this section automatically writes an `app.py` script locally. This script instantiates a Gradio web application UI that processes manual string entries and returns probability metrics generated live by the fine-tuned network.

In [ ]:
# Export application setup code script
gradio_app_code = """
import gradio as gr
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "./fine-tuned-bert-news"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

labels = ["World", "Sports", "Business", "Sci/Tech"]

def classify_news(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    scores = torch.nn.functional.softmax(outputs.logits[0], dim=-1).numpy()
    return {labels[i]: float(scores[i]) for i in range(len(labels))}

interface = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline here..."),
    outputs=gr.Label(num_top_classes=4),
    title="News Topic Classifier Using BERT",
    description="Enter a news headline to predict its topic category (World, Sports, Business, or Sci/Tech)."
)

if __name__ == "__main__":
    interface.launch(share=True)
"""

with open("app.py", "w") as f:
    f.write(gradio_app_code)

print("Deployment file successfully created as 'app.py'.")
print("To run the application inside this notebook environment, simply execute a new cell containing: !python app.py")